# EDA - AvitoTech ML CUP 2026

## Цель соревнования
Предсказать до 160 уникальных item_id для каждого из 94 408 пользователей.

## Метрика
Recall@160 = средняя доля угаданных таргетов по пользователям

## Данные
- train_data/part_*.parquet - 5 млрд событий
- eval_user_events.pq - история eval-пользователей
- item_features.parquet - 178 млн объявлений
- eval_users.csv - 94 408 пользователей
- contact_eids.csv - 11 контактных событий

In [1]:
import polars as pl
import numpy as np
from pathlib import Path

# Пути к данным
DATA_DIR = Path('../data')
EVAL_USERS_PATH = DATA_DIR / 'eval_users.csv'
EVAL_EVENTS_PATH = DATA_DIR / 'eval_user_events.pq'
ITEM_FEATURES_PATH = DATA_DIR / 'item_features.parquet'
CONTACT_EIDS_PATH = DATA_DIR / 'contact_eids.csv'
TRAIN_DIR = DATA_DIR / 'train_data'

print('Готово')

Готово


## 1. Контактные события (contact_eids.csv)

In [2]:
contact_eids_df = pl.read_csv(CONTACT_EIDS_PATH)
print('=== Контактные события ===')
print(f'Количество: {len(contact_eids_df)}')
print(contact_eids_df)

CONTACT_EIDS = contact_eids_df['mapped_eid'].to_list()
print(f'\nСписок: {CONTACT_EIDS}')

=== Контактные события ===
Количество: 11
shape: (11, 1)
┌────────────┐
│ mapped_eid │
│ ---        │
│ i64        │
╞════════════╡
│ 0          │
│ 2          │
│ 4          │
│ 5          │
│ 6          │
│ …          │
│ 9          │
│ 11         │
│ 14         │
│ 15         │
│ 16         │
└────────────┘

Список: [0, 2, 4, 5, 6, 8, 9, 11, 14, 15, 16]


## 2. Eval-пользователи (eval_users.csv)

In [3]:
eval_users = pl.read_csv(EVAL_USERS_PATH)
print('=== Eval-пользователи ===')
print(f'Количество: {len(eval_users):,}')
print(f'Тип: {eval_users["user_id"].dtype}')
print(f'Мин: {eval_users["user_id"].min():,}')
print(f'Макс: {eval_users["user_id"].max():,}')
print('\nПервые 10:')
print(eval_users.head(10))

=== Eval-пользователи ===
Количество: 94,408
Тип: Int64
Мин: 33
Макс: 8,427,689

Первые 10:
shape: (10, 1)
┌─────────┐
│ user_id │
│ ---     │
│ i64     │
╞═════════╡
│ 33      │
│ 63      │
│ 188     │
│ 281     │
│ 295     │
│ 447     │
│ 534     │
│ 547     │
│ 570     │
│ 602     │
└─────────┘


## 3. Фичи объявлений (item_features.parquet)

178 млн объявлений с фичами:
- vertical_id - вертикаль (0-7)
- category_ext_y - категория
- region_id_y, loc_id_y - география
- sid_0..sid_3 - семантические ID (BERT)

In [4]:
items = pl.read_parquet(ITEM_FEATURES_PATH)
print('=== Фичи объявлений ===')
print(f'Всего: {len(items):,}')
print(f'Размер: {items.estimated_size("mb"):.0f} MB')
print('\nКолонки:')
for col, dtype in items.schema.items():
    print(f'  {col}: {dtype}')

print('\n=== vertical_id ===')
vertical_dist = items['vertical_id'].value_counts().sort('vertical_id')
print(vertical_dist)

=== Фичи объявлений ===
Всего: 178,327,659
Размер: 10884 MB

Колонки:
  item_id: UInt32
  vertical_id: UInt32
  category_ext_y: Int64
  region_id_y: Int64
  loc_id_y: Int64
  sid_0_y: Int64
  sid_1_y: Int64
  sid_2_y: Int64
  sid_3_y: Int64

=== vertical_id ===
shape: (8, 2)
┌─────────────┬───────────┐
│ vertical_id ┆ count     │
│ ---         ┆ ---       │
│ u32         ┆ u32       │
╞═════════════╪═══════════╡
│ 0           ┆ 153162868 │
│ 1           ┆ 8609      │
│ 2           ┆ 4947048   │
│ 3           ┆ 6197510   │
│ 4           ┆ 7540650   │
│ 5           ┆ 4143039   │
│ 6           ┆ 14042     │
│ 7           ┆ 2313893   │
└─────────────┴───────────┘


In [5]:
print('=== География ===')
print(f'Регионов: {items["region_id_y"].n_unique():,}')
print(f'Локаций: {items["loc_id_y"].n_unique():,}')
print(f'Категорий: {items["category_ext_y"].n_unique():,}')

print('\nТоп-10 категорий:')
print(items['category_ext_y'].value_counts().sort('count', descending=True).head(10))

=== География ===
Регионов: 85
Локаций: 6,665
Категорий: 52

Топ-10 категорий:
shape: (10, 2)
┌────────────────┬──────────┐
│ category_ext_y ┆ count    │
│ ---            ┆ ---      │
│ i64            ┆ u32      │
╞════════════════╪══════════╡
│ 11             ┆ 32631388 │
│ 1              ┆ 30867536 │
│ 13             ┆ 13342147 │
│ 5              ┆ 8458239  │
│ 4              ┆ 8340775  │
│ 48             ┆ 7540650  │
│ 19             ┆ 7001483  │
│ 14             ┆ 6684641  │
│ 46             ┆ 4947048  │
│ 21             ┆ 4298001  │
└────────────────┴──────────┘


## 4. История eval-пользователей (eval_user_events.pq)

96.8 млн событий от 94 408 пользователей ДО cutoff

In [6]:
eval_events = pl.read_parquet(EVAL_EVENTS_PATH)
print('=== Eval события ===')
print(f'Событий: {len(eval_events):,}')
print(f'Пользователей: {eval_events["user_id"].n_unique():,}')
print(f'Объявлений: {eval_events["item_id"].n_unique():,}')
print(f'Типов событий: {eval_events["eid"].n_unique()}')

print('\nРаспределение eid:')
eid_dist = eval_events['eid'].value_counts().sort('eid')
print(eid_dist)

=== Eval события ===
Событий: 96,807,172
Пользователей: 94,408
Объявлений: 30,465,785
Типов событий: 17

Распределение eid:
shape: (17, 2)
┌─────┬─────────┐
│ eid ┆ count   │
│ --- ┆ ---     │
│ u32 ┆ u32     │
╞═════╪═════════╡
│ 0   ┆ 245     │
│ 1   ┆ 1035910 │
│ 2   ┆ 10635   │
│ 3   ┆ 694334  │
│ 4   ┆ 1575728 │
│ …   ┆ …       │
│ 12  ┆ 276494  │
│ 13  ┆ 180063  │
│ 14  ┆ 337903  │
│ 15  ┆ 750665  │
│ 16  ┆ 104937  │
└─────┴─────────┘


In [7]:
print('=== Статистика по пользователям ===')
user_stats = eval_events.group_by('user_id').agg(
    pl.len().alias('n_events'),
    pl.col('item_id').n_unique().alias('n_items')
)
print('\nСобытий на пользователя:')
print(user_stats['n_events'].describe())
print('\nОбъявлений на пользователя:')
print(user_stats['n_items'].describe())

=== Статистика по пользователям ===

Событий на пользователя:
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 94408.0     │
│ null_count ┆ 0.0         │
│ mean       ┆ 1025.412804 │
│ std        ┆ 1846.702506 │
│ min        ┆ 1.0         │
│ 25%        ┆ 82.0        │
│ 50%        ┆ 385.0       │
│ 75%        ┆ 1185.0      │
│ max        ┆ 59451.0     │
└────────────┴─────────────┘

Объявлений на пользователя:
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 94408.0     │
│ null_count ┆ 0.0         │
│ mean       ┆ 609.492532  │
│ std        ┆ 1070.519147 │
│ min        ┆ 1.0         │
│ 25%        ┆ 49.0        │
│ 50%        ┆ 233.0       │
│ 75%        ┆ 715.0       │
│ max        ┆ 25876.0     │
└────────────┴─────────────┘


## 5. Train данные (part_*.parquet)

5 млрд событий, 100 партиций по user_id % 100

In [8]:
print('=== Train данные (part_000) ===')
train = pl.read_parquet(TRAIN_DIR / 'part_000.parquet')
print(f'Строк: {len(train):,}')
print(f'Пользователей: {train["user_id"].n_unique():,}')
print(f'Объявлений: {train["item_id"].n_unique():,}')
print(f'\nВремя: {train["timestamp"].min():,} - {train["timestamp"].max():,}')

=== Train данные (part_000) ===
Строк: 51,682,001
Пользователей: 82,570
Объявлений: 19,708,019

Время: 1,768,953,600,000 - 1,776,211,199,000


## 6. Кросс-анализ

In [9]:
print('=== Вертикали в eval событиях ===')
events_v = eval_events.join(
    items.select(['item_id', 'vertical_id']),
    on='item_id',
    how='inner'
)
print(events_v['vertical_id'].value_counts().sort('vertical_id'))

=== Вертикали в eval событиях ===
shape: (8, 2)
┌─────────────┬──────────┐
│ vertical_id ┆ count    │
│ ---         ┆ ---      │
│ u32         ┆ u32      │
╞═════════════╪══════════╡
│ 0           ┆ 60950015 │
│ 1           ┆ 20603    │
│ 2           ┆ 5642208  │
│ 3           ┆ 10278971 │
│ 4           ┆ 2931780  │
│ 5           ┆ 16233082 │
│ 6           ┆ 13170    │
│ 7           ┆ 737087   │
└─────────────┴──────────┘


In [10]:
print('=== Контактные события ===')
contact_events = eval_events.filter(pl.col('eid').is_in(CONTACT_EIDS))
print(f'Всего контактов: {len(contact_events):,}')
print(f'Пользователей: {contact_events["user_id"].n_unique():,}')
print(f'Объявлений: {contact_events["item_id"].n_unique():,}')

=== Контактные события ===
Всего контактов: 4,043,027
Пользователей: 81,504
Объявлений: 2,496,868
